# 10 · Profile: ACS PUMS, what NYC graduates actually do and earn

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/acs_pums.md`.

**Why this source matters.** Every other source in this project is about jobs (how many, what they pay, what they need) or degrees (how many each college awards). None of them follows a *person*. The Census Bureau's American Community Survey Public Use Microdata Sample (ACS PUMS) does: one row is one real (anonymized) person, with their degree level, college major, job, industry, borough, hours, and pay on the same row. It's the only government source that can say *what NYC residents with a given major actually do for a living*. The brief uses it three ways:
1. Check the major → job links (decision D2, nb 06): do NYC graduates with major X actually work in the jobs the CIP→SOC crosswalk says?
2. **The crowded-field rule** (brief S11): how many young bachelor's holders work in jobs that don't need a bachelor's (underemployment).
3. **Junior vs senior** (brief O1): what share of each job's workers are graduation-age.

**What this notebook does:** streams the New York State person file once, keeps NYC, checks grain, codes, weights, and ties to the Census Bureau's published tables. It ends with an issues list and a proposed staging table. It changes nothing.

In [1]:
import re
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
BOROUGHS = {"005": "Bronx", "047": "Brooklyn", "061": "Manhattan", "081": "Queens", "085": "Staten Island"}

man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key.isin(["acs_pums", "acs_summary_tables"])][["key", "file", "downloaded_utc", "bytes", "note"]]
display(FILES)
P = {Path(f).name: RAW / f for f in FILES.file}

,key,file,downloaded_utc,bytes,note
91,acs_pums,outcomes/acs_pums/csv_pny_2020-2024_5yr.zip,2026-10-12T17:09:02+00:00,140098497,ACS PUMS NY persons 2020-2024 (major x job x age x pay)
92,acs_pums,outcomes/acs_pums/PUMS_Data_Dictionary_2020-2024.csv,2026-10-12T17:09:03+00:00,330431,
93,acs_pums,outcomes/acs_pums/2024_ACS_Code_Lists.xlsx,2026-10-12T17:09:04+00:00,257505,Field-of-degree (FOD1P) code list
94,acs_pums,outcomes/acs_pums/2018-occupation-code-list-and-crosswalk.xlsx,2026-10-12T17:09:05+00:00,160312,Census OCCP -> SOC
95,acs_pums,outcomes/acs_pums/2020_Census_Tract_to_2020_PUMA.txt,2026-10-12T17:09:07+00:00,1709076,Defines the 55 NYC PUMAs
194,acs_summary_tables,outcomes/acs_summary_tables/acsdt5y2024-b01003.dat,2026-10-13T03:21:35+00:00,18313708,ACS 2020-2024 5-year published table: Total population
195,acs_summary_tables,outcomes/acs_summary_tables/acsdt5y2024-b15003.dat,2026-10-13T03:21:36+00:00,92292840,"ACS 2020-2024 5-year published table: Educational attainment, age 25+"
196,acs_summary_tables,outcomes/acs_summary_tables/acsdt5y2024-b15012.dat,2026-10-13T03:21:38+00:00,63236635,ACS 2020-2024 5-year published table: Bachelor's field of degree (...


## 0. Source header

| | ACS 2020–2024 5-year PUMS, New York persons (`acs_pums`) |
|---|---|
| **Publisher** | U.S. Census Bureau, American Community Survey Office |
| **Files** | `csv_pny_2020-2024_5yr.zip` (`psam_p36.csv` = NY persons + a README), the data dictionary, the 2024 ACS code lists (field of degree, occupation, industry), the Census occupation → SOC list, and the 2020 tract → PUMA file (to find NYC). Tie-out: three published 5-year tables (`acs_summary_tables`: B01003 population, B15003 education, B15012 bachelor's fields) |
| **Release** | 2020–2024 5-year file, README dated 2026-03-05 (file dated 2026-01-16) |
| **How it was collected** | **Survey**. About 3.5M addresses a year nationwide; the PUMS is a subsample of the responses (roughly two thirds), anonymized: the smallest place is a PUMA (≈100k+ people), ages are top-coded, pay is rounded/top-coded, and some answers are swapped. Missing answers are imputed ("allocated") and flagged |
| **Weights** | Every person has a weight `PWGTP` (how many people they stand for) and 80 replicate weights `PWGTP1–80` for margins of error (successive differences replication, SDR). The README says 2024 is the last year with design factors; Census recommends replicate weights |
| **5-year pooling** | 2020–2024 responses pooled. Weights are scaled so the whole file represents the period average population (don't divide by 5). Dollar amounts are in each response year's dollars; `ADJINC` brings them to 2024 dollars |
| **License** | Public domain |
| **Documentation** | [PUMS documentation](https://www.census.gov/programs-surveys/acs/microdata/documentation.html) (User Guide, Accuracy of the Data), data dictionary in `data/raw/outcomes/acs_pums/` |

## 1. Load as text

The zip holds one 640 MB CSV for all of New York State. Read it in chunks, count everything, keep two groups: NYC residents (home PUMA is one of NYC's 55) and NY State residents who work in NYC (for the commuter check in §5).

In [2]:
# NYC = the PUMAs whose tracts are in the five borough counties (2020 PUMA definitions)
tr = pd.read_csv(P["2020_Census_Tract_to_2020_PUMA.txt"], dtype=str, encoding="utf-8-sig")
nyc_tr = tr[(tr.STATEFP == "36") & tr.COUNTYFP.isin(BOROUGHS)]
PUMA_CTY = dict(zip(nyc_tr.PUMA5CE, nyc_tr.COUNTYFP))
print(f"{len(PUMA_CTY)} NYC PUMAs:", {BOROUGHS[c]: n for c, n in pd.Series(PUMA_CTY).value_counts().sort_index().items()})
print("A PUMA never crosses a borough line:", nyc_tr.groupby("PUMA5CE").COUNTYFP.nunique().max() == 1)

DICT = pd.read_csv(P["PUMS_Data_Dictionary_2020-2024.csv"], header=None, names=range(7), dtype=str, keep_default_na=False)  # NAME rows 5 fields, VAL rows 7
LABEL = DICT[DICT[0] == "NAME"].set_index(1)[4].to_dict()

zf = zipfile.ZipFile(P["csv_pny_2020-2024_5yr.zip"])
print("Files in the zip:", zf.namelist())
hdr = pd.read_csv(zf.open("psam_p36.csv"), nrows=0).columns
groups = {"replicate weights PWGTP1-80": hdr.str.fullmatch(r"PWGTP\d+").sum(),
          "allocation flags F*P": hdr.str.fullmatch(r"F\w+P|F\w+C").sum()}
print(f"{len(hdr)} columns: {groups}, {len(hdr) - sum(groups.values())} variables")

55 NYC PUMAs: {'Bronx': 10, 'Brooklyn': 18, 'Manhattan': 10, 'Queens': 14, 'Staten Island': 3}
A PUMA never crosses a borough line: True
Files in the zip: ['psam_p36.csv', 'ACS2020_2024_PUMS_README.pdf']
285 columns: {'replicate weights PWGTP1-80': np.int64(80), 'allocation flags F*P': np.int64(78)}, 127 variables


In [3]:
NYC_POW = {"04100": "Manhattan", "04200": "Bronx", "04300": "Brooklyn", "04400": "Queens", "04500": "Staten Island"}
USE = ["SERIALNO", "SPORDER", "PUMA", "ADJINC", "PWGTP", "AGEP", "SEX", "SCHL", "SCHG", "FOD1P", "FOD2P", "SCIENGP",
       "ESR", "COW", "OCCP", "SOCP", "INDP", "NAICSP", "WAGP", "PERNP", "WKHP", "WKWN", "POWSP", "POWPUMA",
       "FSCHLP", "FFODP", "FOCCP", "FINDP", "FWAGP", "FCOWP"] + [f"PWGTP{i}" for i in range(1, 81)]

n_state, keep = 0, []
for ch in pd.read_csv(zf.open("psam_p36.csv"), dtype=str, keep_default_na=False, usecols=USE, chunksize=400_000):
    n_state += len(ch)
    keep.append(ch[ch.PUMA.isin(PUMA_CTY) | ((ch.POWSP == "036") & ch.POWPUMA.isin(NYC_POW))])
raw = pd.concat(keep, ignore_index=True)
raw["resident"] = raw.PUMA.isin(PUMA_CTY)
print(f"NY State person rows: {n_state:,} | kept: {len(raw):,} "
      f"({raw.resident.sum():,} NYC residents + {(~raw.resident).sum():,} NY State residents working in NYC)")

NY State person rows: 984,474 | kept: 347,371 (326,729 NYC residents + 20,642 NY State residents working in NYC)


**Load findings**
- The zip holds one person file for all of New York State (`psam_p36.csv`, 640 MB) + a 2-page README. It streams in about 10 seconds.
- **285 columns** = 127 variables + 78 allocation flags (`F…P`, "this answer was imputed") + 80 replicate weights.
- NYC = 55 PUMAs (2020 definitions): Bronx 10, Brooklyn 18, Manhattan 10, Queens 14, Staten Island 3. No PUMA crosses a borough line, so borough is exact; nothing smaller than a PUMA exists.
- 984,474 NY State rows → 326,729 NYC residents + 20,642 NY State suburbanites who work in NYC.

## 2. What one row is (grain)

**Presumed key:** `SERIALNO` (household or group-quarters unit) + `SPORDER` (person number in it). The response year is the first four characters of `SERIALNO`; `GQ` in it marks group quarters (dorms, nursing homes, jails).

In [4]:
print("Duplicates on SERIALNO + SPORDER:", raw.duplicated(["SERIALNO", "SPORDER"]).sum())
raw["year"] = raw.SERIALNO.str[:4]
raw["gq"] = raw.SERIALNO.str[4:6] == "GQ"
raw["w"] = raw.PWGTP.astype(int)
raw["age"] = raw.AGEP.astype(int)
r = raw[raw.resident].copy()
r["borough"] = r.PUMA.map(PUMA_CTY).map(BOROUGHS)

t = r.groupby("year").agg(rows=("w", "size"), weighted=("w", "sum"), gq_rows=("gq", "sum"))
t["adjinc"] = r.groupby("year").ADJINC.unique().map(lambda a: ", ".join(a))
t["to_2024_dollars"] = t.adjinc.astype(int) / 1e6
display(t)
print(f"All years together: {len(r):,} rows standing for {r.w.sum():,} people (period average)")
print("Weights: min", r.w.min(), "| median", int(r.w.median()), "| max", r.w.max(), "| zero weights:", (r.w == 0).sum())

Duplicates on SERIALNO + SPORDER: 0


,rows,weighted,gq_rows,adjinc,to_2024_dollars
year,,,,,
2020,58688,1747567,4593,1222017,1.222017
2021,67275,1688355,3597,1193241,1.193241
2022,66865,1679368,3865,1117193,1.117193
2023,66390,1654651,4256,1049470,1.049470
2024,67511,1710284,4837,1015250,1.015250


All years together: 326,729 rows standing for 8,480,225 people (period average)
Weights: min 1 | median 18 | max 922 | zero weights: 0


**Grain findings**
- Key `SERIALNO + SPORDER` is unique . One row = one person in one response year.
- The year is only inside `SERIALNO` (first 4 characters). 58.7k–67.5k NYC rows a year; 2020 has fewer rows (pandemic year: the Bureau collected less and adjusted the weights).
- Weights sum to 8,480,225 across all five years: the period-average population. A single year's rows sum to about a fifth of that, so never filter to one year and read the total as that year's population.
- `ADJINC` is one value per year (1.222 for 2020 → 1.015 for 2024). Multiply `WAGP` by `ADJINC / 1,000,000` to get 2024 dollars.
- Group quarters (dorms, nursing homes, jails) are 3.6–4.8k rows a year; they have person weights like everyone else.

## 3. Column profile

In PUMS a blank is almost never "missing": it means not in universe (the question doesn't apply, e.g. no job code for a child). Real non-answers are filled in by Census (allocated) and flagged in the matching `F…P` column.

In [5]:
COLS = ["AGEP", "SCHL", "SCHG", "FOD1P", "FOD2P", "ESR", "COW", "OCCP", "SOCP", "INDP", "NAICSP", "WAGP", "WKHP", "WKWN", "POWPUMA"]
prof = pd.DataFrame({
    "label": [LABEL.get(c, "")[:60] for c in COLS],
    "blank_%": [(r[c] == "").mean() * 100 for c in COLS],
    "distinct": [r[c].nunique() for c in COLS],
    "top": [", ".join(r[c].value_counts().head(3).index) for c in COLS],
}, index=COLS).round(1)
display(prof)

# Universe checks: field of degree exists exactly for bachelor's or higher; job code exactly for people who worked in the last 5 years
print("FOD1P filled, by education (SCHL):", r.assign(f=r.FOD1P != "").groupby(r.SCHL >= "21").f.mean().rename({False: "below bachelor's", True: "bachelor's+"}).to_dict())
print("SOCP filled, by employment status:", r.assign(f=r.SOCP != "").groupby("ESR").f.mean().round(3).to_dict())

# How much is imputed? (weighted, among the people the question applies to)
def alloc(flag, universe):
    u = r[universe]; return round((u.w * (u[flag] == "1")).sum() / u.w.sum() * 100, 1)
emp_mask = r.ESR.isin(["1", "2", "4", "5"])
print("% allocated (imputed by Census): "
      f"education {alloc('FSCHLP', r.age >= 3)} | field of degree {alloc('FFODP', r.SCHL >= '21')} | "
      f"occupation {alloc('FOCCP', emp_mask)} | industry {alloc('FINDP', emp_mask)} | wages {alloc('FWAGP', emp_mask)}")

,label,blank_%,distinct,top
AGEP,Age,0.0,93,"30, 31, 29"
SCHL,Educational attainment,2.5,25,"21, 16, 22"
SCHG,Grade level attending,78.4,17,", 15, 16"
FOD1P,Recoded field of degree - first entry,66.5,175,", 5200, 3301"
FOD2P,Recoded field of degree - second entry,95.8,166,", 5501, 5200"
ESR,Employment status recode,14.5,7,"1, 6,"
COW,Class of worker,39.2,10,", 1, 2"
OCCP,Occupation recode for 2018 and later based on 2018 OCC codes,39.2,530,", 3601, 0440"
SOCP,Standard Occupational Classification (SOC) codes for 2018 an,39.2,530,", 311121, 1191XX"
INDP,Industry recode for 2023 and later based on 2022 IND codes,39.2,262,", 7860, 8680"


FOD1P filled, by education (SCHL): {"below bachelor's": 0.0, "bachelor's+": 1.0}
SOCP filled, by employment status: {'': 0.0, '1': 1.0, '2': 1.0, '3': 1.0, '4': 1.0, '5': 1.0, '6': 0.268}


% allocated (imputed by Census): education 15.5 | field of degree 25.2 | occupation 24.4 | industry 23.3 | wages 34.1


**Profile findings**
- Blanks are "not in universe", and they follow the rules exactly: field of degree is filled for 100% of bachelor's+ and 0% below; a job code is filled for everyone employed or unemployed and for 27% of people not in the labor force (they worked in the last 5 years).
- **A lot is imputed.** Weighted share filled in by Census: education 15.5%, field of degree 25.2%, occupation 24.4%, industry 23.3%, wages 34.1%. That's normal for ACS, but it means one in four "major → job" pairs in PUMS is partly modeled, not reported. Keep the flags.
- `NAICSP` mixes levels and letters (`722Z`, `622M`, `611M1`, `3M`, `4M`): Census groups, not NAICS codes.

## 4. Codes, hierarchy and special values

### 4a. Education and field of degree

In [6]:
SCHL_LBL = DICT[(DICT[0] == "VAL") & (DICT[1] == "SCHL")].set_index(5)[6].to_dict()
deg = r[r.age >= 25].groupby("SCHL").w.sum()
print("Age 25+, top education levels (weighted):", {SCHL_LBL[k]: f"{v:,}" for k, v in deg.loc["20":].items()})

fod = pd.read_excel(P["2024_ACS_Code_Lists.xlsx"], sheet_name="Field of Degree", header=None, dtype=str).iloc[2:]
fod.columns = ["title", "code"]
fod["group"] = fod.title.where(fod.code.isna()).ffill()
FOD = fod.dropna(subset=["code"]).set_index("code").title.str.strip().to_dict()
print(f"\nField-of-degree code list: {len(FOD)} codes | in NYC data: {r.FOD1P[r.FOD1P != ''].nunique()} "
      f"| NYC codes not on the list: {sorted(set(r.FOD1P) - set(FOD) - {''})}")
print("Has a second major:", round((r[r.FOD1P != ''].w * (r.FOD2P != '')).sum() / r[r.FOD1P != ''].w.sum() * 100, 1), "% of bachelor's+")
top = r[r.FOD1P != ""].groupby("FOD1P").w.sum().sort_values(ascending=False).head(12)
display(pd.DataFrame({"field": top.index.map(FOD), "nyc_residents_bachelors_plus": top.values}, index=top.index))

Age 25+, top education levels (weighted): {"Associate's degree": '393,412', "Bachelor's degree": '1,458,884', "Master's degree": '765,746', "Professional degree beyond a bachelor's degree": '203,585', 'Doctorate degree': '107,372'}



Field-of-degree code list: 196 codes | in NYC data: 174 | NYC codes not on the list: []


Has a second major: 12.4 % of bachelor's+


,field,nyc_residents_bachelors_plus
FOD1P,,
5200,Psychology,138266
5501,Economics,121768
3301,English Language And Literature,115570
6201,Accounting,111944
6203,Business Management And Administration,106708
2300,General Education,88881
6200,General Business,87407
5506,Political Science And Government,86993
6107,Nursing,85598


**Field-of-degree findings**
- Field of degree exists only for bachelor's and higher. ACS asks the major question only of people with a bachelor's. Associate's holders (393k NYC residents 25+) have no major at all. So ACS can check major → job links only for bachelor's degrees, never for CUNY's associate programs.
- `FOD1P` codes: 196 on the 2024 code list, 174 used in NYC, every NYC code is on the list. 12.4% of bachelor's+ holders also have a second major (`FOD2P`).
- The field is the bachelor's field even for people with a graduate degree (a psychology BA with an MBA is still `5200`).
- Biggest NYC fields (bachelor's+, all ages): Psychology, Economics, English, Accounting, Business Management.

### 4b. Jobs: Census occupation codes → SOC

ACS doesn't code jobs in SOC directly. It uses ~570 Census occupation codes (`OCCP`) and gives a SOC-style code (`SOCP`) for each. Where Census can't tell detailed jobs apart, `SOCP` is a group: a SOC broad group (`252020` = 25-2020 elementary and middle school teachers), or a Census-made group with `X` or `Y` (`1191XX` = several "other managers"). Can each `SOCP` find a BLS "typical entry education" (nb 04), which the underemployment measure needs?

In [7]:
ed = pd.read_excel(RAW / man.loc[man.key == "bls_ep_education", "file"].iloc[0], sheet_name="Table 5.4", header=1, dtype=str, keep_default_na=False)
ed = ed[ed.iloc[:, 1].str.fullmatch(r"\d\d-\d{4}")]
ENTRY = dict(zip(ed.iloc[:, 1], ed.iloc[:, 2]))   # BLS EP codes: SOC 2018 detailed + a few BLS aggregates

def soc_prefix(socp):
    # 252020 -> "25202", 1191XX -> "1191": every BLS code starting with it sits under that group
    return re.sub(r"0+$", "", socp.replace("X", "").replace("Y", ""))

def resolve(socp):
    hits = {c: e for c, e in ENTRY.items() if c.replace("-", "").startswith(soc_prefix(socp))}
    parent = f"{socp[:2]}-{socp[2:5]}0"   # BLS publishes a few jobs only as their broad group (31-1120, 13-1020, 21-1018 ...)
    if f"{socp[:2]}-{socp[2:]}" in ENTRY: kind = "exact"
    elif not hits and parent in ENTRY: kind, hits = "BLS publishes the parent group", {parent: ENTRY[parent]}
    elif not hits and socp[:5] + "8" in [c.replace("-", "")[:6] for c in ENTRY if c.endswith("8")]:
        c8 = next(c for c in ENTRY if c.replace("-", "") == socp[:5] + "8"); kind, hits = "BLS publishes the parent group", {c8: ENTRY[c8]}
    elif not hits: kind = "no match"
    elif len(set(hits.values())) == 1: kind = "group, one entry level"
    else: kind = "group, mixed entry levels"
    return pd.Series({"kind": kind, "n_bls_codes": len(hits), "entry": next(iter(set(hits.values()))) if len(set(hits.values())) == 1 else None})

emp = r[emp_mask].copy()
SOCMAP = pd.Series(emp.SOCP.unique()).to_frame("SOCP").set_index("SOCP").index.to_series().apply(resolve)
emp = emp.join(SOCMAP, on="SOCP")
k = emp.groupby("kind").agg(socp_codes=("SOCP", "nunique"), workers=("w", "sum"))
k["share_%"] = (k.workers / k.workers.sum() * 100).round(1)
display(k)
for kind in ["group, mixed entry levels", "no match"]:
    s = emp[emp.kind == kind].groupby("SOCP").w.sum().sort_values(ascending=False).head(8)
    print(f"{kind}, biggest:", dict(s))

,socp_codes,workers,share_%
kind,,,
BLS publishes the parent group,7,219280,5.4
exact,370,2517594,62.4
"group, mixed entry levels",77,739728,18.3
"group, one entry level",66,554181,13.7
no match,4,2177,0.1


group, mixed entry levels, biggest: {'1191XX': np.int64(113919), '2310XX': np.int64(70838), '533030': np.int64(59272), '259040': np.int64(57720), '251000': np.int64(45307), '352010': np.int64(38554), '119030': np.int64(31126), '131070': np.int64(24296)}
no match, biggest: {'559830': np.int64(1431), '553010': np.int64(577), '552010': np.int64(127), '551010': np.int64(42)}


**Job-code findings** (weighted by employed NYC residents)

| How `SOCP` reaches a BLS entry-education level | Share of workers |
|---|---|
| **Exact** detailed SOC 2018 code | **62.4%** |
| Group, but every job under it has the same entry level | 13.7% |
| BLS publishes only the parent group (`311121` home health aides → BLS `31-1120`; buyers → `13-1020`; counselors → `21-1018`) | 5.4% |
| **Group with mixed entry levels** (`1191XX` other managers, `2310XX` lawyers + judges, `533030` truck drivers, `259040` teaching assistants, `251000` postsecondary teachers) | **18.3%** |
| No match (military `55xxxx`) | 0.1% |

So 81.6% of workers get one clear entry level and 18.3% need a rule (pick the level of the job with the most NYC employment under the group, using OEWS NY metro, nb 03). This is the same set of BLS aggregates nb 06 found.

### 4c. Industry, and stability of codes across the five years

In [8]:
print("NAICSP examples:", r.NAICSP[r.NAICSP != ""].value_counts().head(8).to_dict())
print("NAICSP 2-digit sectors:", sorted(r.NAICSP[r.NAICSP != ""].str[:2].unique()))
for col in ["OCCP", "INDP", "FOD1P"]:
    yrs = r[r[col] != ""].groupby(col).year.nunique()
    gone = r[r[col].isin(yrs[yrs < 5].index)]
    print(f"{col}: {len(yrs)} codes, {(yrs < 5).sum()} not seen in every year, holding {gone.w.sum() / r[r[col] != ''].w.sum():.2%} of the weight")

NAICSP examples: {'6111': 12124, '722Z': 11789, '622M': 10044, '23': 8787, '6216': 6802, '611M1': 6598, '6241': 5728, '5415': 4438}
NAICSP 2-digit sectors: ['11', '21', '22', '23', '31', '32', '33', '3M', '42', '44', '45', '48', '49', '4M', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81', '92', '99']


OCCP: 529 codes, 60 not seen in every year, holding 0.25% of the weight


INDP: 261 codes, 33 not seen in every year, holding 0.11% of the weight


FOD1P: 174 codes, 11 not seen in every year, holding 0.05% of the weight


**Industry / stability findings**
- `NAICSP` starts with a 2-digit NAICS sector for every code, plus Census groups `3M` (manufacturing, not specified) and `4M` (wholesale/retail, not specified). It maps to the QCEW sectors (nb 07) by its first 2 characters, with `3M → 31-33` and `4M` needing a rule.
- No code-system break inside the five years: occupation (2018 Census codes), industry (2022 codes, recoded for all years) and field of degree. Codes missing in some years are rare jobs, holding 0.05–0.25% of the weight.

## 5. Coverage: who is in the NYC slice

### 5a. Boroughs, and where people work

In [9]:
print("Residents by borough (weighted):", r.groupby("borough").w.sum().to_dict())
wk = raw[raw.ESR.isin(["1", "2", "4", "5"]) & (raw.POWSP == "036") & raw.POWPUMA.isin(NYC_POW)]
print(f"\nWorkers whose job is in NYC, in the NY State file: {wk.w.sum():,}; "
      f"NYC residents {wk[wk.resident].w.sum() / wk.w.sum():.1%}, NY suburbs {wk[~wk.resident].w.sum() / wk.w.sum():.1%}")
out = emp[~emp.POWPUMA.isin(NYC_POW)]
print(f"Employed NYC residents who work outside NYC (or from no fixed place): {out.w.sum() / emp.w.sum():.1%}")
print("Place-of-work state for those:", out.groupby("POWSP").w.sum().sort_values(ascending=False).head(5).to_dict())

Residents by borough (weighted): {'Bronx': 1403853, 'Brooklyn': 2630588, 'Manhattan': 1628963, 'Queens': 2322025, 'Staten Island': 494796}



Workers whose job is in NYC, in the NY State file: 4,116,550; NYC residents 89.1%, NY suburbs 10.9%
Employed NYC residents who work outside NYC (or from no fixed place): 9.0%
Place-of-work state for those: {'036': 161641, '': 117686, '034': 59699, '009': 7595, '042': 3580}


**Geography findings**
- The file counts where people live, not where they work. 89.1% of NYC jobs held by NY State residents are held by NYC residents, 10.9% by suburban New Yorkers. New Jersey and Connecticut commuters aren't in this file at all (they're in `psam_p34` / `psam_p09`). So PUMS can't count "everyone who works in NYC" without two more state files.
- 9.0% of employed NYC residents work outside NYC (mostly NY suburbs and NJ) or have no fixed workplace.
- **For this project that's fine:** the question is what NYC graduates do, and CUNY graduates mostly live in NYC. The staging table uses NYC residents. Label it "NYC residents" on every visual.

### 5b. The graduation-age slice: how many people, how many survey rows?

The New York Fed's "recent college graduates" are ages 22–27 with a bachelor's (and no higher degree). We use the same window for both degree levels. What matters for a survey is the unweighted row count: every estimate rests on those rows.

In [10]:
r["degree"] = np.select([r.SCHL == "20", r.SCHL == "21", r.SCHL >= "22"], ["associate", "bachelor", "graduate"], "below associate")
g = r[r.age.between(22, 27) & r.degree.isin(["associate", "bachelor"])].copy()
g["enrolled"] = g.SCHG != ""
g["employed"] = g.ESR.isin(["1", "2", "4", "5"])
s = g.groupby("degree").apply(lambda x: pd.Series({
    "survey_rows": len(x), "people": x.w.sum(),
    "in_school_now_%": (x.w * x.enrolled).sum() / x.w.sum() * 100,
    "employed_%": (x.w * x.employed).sum() / x.w.sum() * 100}), include_groups=False).round(1)
display(s)

b = g[g.degree == "bachelor"]
n = b.groupby("FOD1P").size()
cuts = pd.Series({f"fields with >= {c} rows": int((n >= c).sum()) for c in [30, 50, 100]})
print(f"Bachelor's 22-27: {len(n)} fields of degree; median {n.median():.0f} rows per field;", cuts.to_dict())
print(f"Share of these graduates in fields with >= 30 rows: {b[b.FOD1P.isin(n[n >= 30].index)].w.sum() / b.w.sum():.1%}")

,survey_rows,people,in_school_now_%,employed_%
degree,,,,
associate,1727.0,47292.0,40.8,63.4
bachelor,12346.0,303668.0,23.0,80.3


Bachelor's 22-27: 158 fields of degree; median 28 rows per field; {'fields with >= 30 rows': 77, 'fields with >= 50 rows': 55, 'fields with >= 100 rows': 31}
Share of these graduates in fields with >= 30 rows: 93.9%


**Graduation-age findings** (NYC residents, 22–27, highest degree = associate's or bachelor's)

| | Survey rows | People (weighted) | Still in school | Employed |
|---|---|---|---|---|
| Associate's | 1,727 | 47,292 | **40.8%** | 63.4% |
| Bachelor's | 12,346 | 303,668 | 23.0% | 80.3% |

- Four in ten young associate's holders are still in school: direct evidence for the "transfer pathway" label (D2 / Liberal Arts).
- Bachelor's 22–27 spread over 158 fields, but the median field has only 28 survey rows. 77 fields have ≥ 30 rows, 31 have ≥ 100. The fields with ≥ 30 rows hold 93.9% of the graduates, so a ≥ 30-row floor loses few people but many small fields.

## 6. Relationships: will it join to the other sources?

In [11]:
# Job: SOCP -> SOC 2018 (nb 06 reference list) and the BLS entry-education table (nb 04), weighted by workers
soc18 = pd.read_excel(RAW / "jobs/bls_soc_structure/soc_structure_2018.xlsx", header=None, dtype=str).stack().str.strip()
SOC18 = set(soc18[soc18.str.fullmatch(r"\d\d-\d{4}")])
emp["soc_fmt"] = emp.SOCP.str[:2] + "-" + emp.SOCP.str[2:]
tot = emp.w.sum()
print(f"Employed residents whose SOCP is a SOC 2018 code (any level): {emp[emp.soc_fmt.isin(SOC18)].w.sum() / tot:.1%}; "
      f"a detailed SOC 2018 code: {emp[emp.soc_fmt.isin(SOC18) & ~emp.SOCP.str.endswith('0')].w.sum() / tot:.1%}; "
      f"has X/Y: {emp[emp.SOCP.str.contains('X|Y')].w.sum() / tot:.1%}")
print(f"... resolves to one BLS entry level (exact or via group): {emp.entry.notna().mul(emp.w).sum() / tot:.1%}")

# Census's own occupation list: OCCP -> SOC, the rule behind SOCP
occ = pd.read_excel(P["2018-occupation-code-list-and-crosswalk.xlsx"], sheet_name="2018 Census Occ Code List", header=None, dtype=str).iloc[:, 1:4]
occ.columns = ["title", "occp", "soc"]
occ = occ[occ.occp.str.fullmatch(r"\d{4}", na=False)]
print(f"\nCensus list: {len(occ)} OCCP codes; NYC OCCP codes not on it: {sorted(set(r.OCCP) - set(occ.occp) - {''})}")
pairs = emp.groupby("OCCP").SOCP.nunique()
print("Each OCCP has exactly one SOCP in the data:", (pairs == 1).all())

# Industry: NAICSP first 2 digits -> the NAICS sectors used by QCEW (nb 07)
print("NAICSP sectors (first 2 chars):", emp.NAICSP.str[:2].value_counts().head(25).to_dict())

# Major: no FOD1P -> CIP crosswalk is published
print(f"\nField of degree -> CIP: no crosswalk in data/raw/. FOD1P has {len(FOD)} codes; IPEDS uses ~1,600 6-digit CIP 2020 codes (nb 05).")

Employed residents whose SOCP is a SOC 2018 code (any level): 86.7%; a detailed SOC 2018 code: 67.6%; has X/Y: 13.2%
... resolves to one BLS entry level (exact or via group): 81.6%

Census list: 570 OCCP codes; NYC OCCP codes not on it: []
Each OCCP has exactly one SOCP in the data: True
NAICSP sectors (first 2 chars): {'62': 29808, '54': 17581, '61': 16665, '52': 10224, '72': 9707, '81': 7952, '45': 7453, '23': 6784, '48': 6744, '92': 6471, '51': 6161, '56': 5861, '44': 4890, '71': 4191, '53': 3802, '42': 2580, '49': 2477, '33': 1729, '31': 1469, '32': 1185, '4M': 709, '22': 618, '55': 253, '3M': 220, '11': 142}

Field of degree -> CIP: no crosswalk in data/raw/. FOD1P has 196 codes; IPEDS uses ~1,600 6-digit CIP 2020 codes (nb 05).


**Relationship findings**
- **Job:** 86.7% of workers have a code that exists in SOC 2018 (detailed or group); 67.6% a detailed one; 13.2% a Census `X`/`Y` group. With the group rules in §4b, 81.6% reach a BLS entry level. Every NYC `OCCP` is on the Census 2018 list, and each `OCCP` has exactly one `SOCP`.
- **Industry:** first 2 characters of `NAICSP` = QCEW sector (nb 07), with the `3M`/`4M` exceptions.
- **Major:** no published crosswalk from ACS field of degree to CIP. Census has one internally (an IPUMS forum post says it was sent on request), but it isn't public. brief O5's hand-built FOD → CIP table is needed to compare ACS majors with IPEDS awards (nb 05). FOD is coarse (174 codes in NYC vs ~1,600 CIP codes), so the table maps each FOD to a set of CIP 4-digit series, many-to-many.

## 7. Tie-out: PUMS vs the Census Bureau's published 2020–2024 tables

The published tables use the full ACS sample; PUMS is a subsample. Population is controlled to the same estimates, so it should match closely; education and field of degree should be near, not exact.

In [12]:
def published(table):
    t = pd.read_csv(P[f"acsdt5y2024-{table}.dat"], sep="|", dtype=str)
    t = t[t.GEO_ID.isin([f"0500000US36{c}" for c in BOROUGHS])].set_index("GEO_ID")
    t.index = t.index.str[-3:].map(BOROUGHS)
    return t

b01, b03, b12 = published("b01003"), published("b15003"), published("b15012")
pop = pd.DataFrame({"published": b01.B01003_E001.astype(int), "pums": r.groupby("borough").w.sum()})
ba_cols = [f"B15003_E0{i}" for i in (22, 23, 24, 25)]
pop["pub_bachelors_plus_25"] = b03[ba_cols].astype(int).sum(axis=1)
pop["pums_bachelors_plus_25"] = r[(r.age >= 25) & (r.SCHL >= "21")].groupby("borough").w.sum()
pop["pub_associate_25"] = b03.B15003_E021.astype(int)
pop["pums_associate_25"] = r[(r.age >= 25) & (r.SCHL == "20")].groupby("borough").w.sum()
pop.loc["NYC"] = pop.sum()
for a, bb in [("population", ("published", "pums")), ("bachelors+", ("pub_bachelors_plus_25", "pums_bachelors_plus_25")), ("associate", ("pub_associate_25", "pums_associate_25"))]:
    pop[f"diff_%_{a}"] = ((pop[bb[1]] / pop[bb[0]] - 1) * 100).round(2)
display(pop)

ba25 = r[(r.age >= 25) & (r.FOD1P != "")]
print(f"B15012 'fields of bachelor's degrees reported' NYC: {b12.B15012_E001.astype(int).sum():,} | "
      f"PUMS first majors {ba25.w.sum():,} | first + second majors {ba25.w.sum() + (ba25.w * (ba25.FOD2P != '')).sum():,}")

,published,pums,pub_bachelors_plus_25,pums_bachelors_plus_25,pub_associate_25,pums_associate_25,diff_%_population,diff_%_bachelors+,diff_%_associate
Bronx,1404779,1403853,208354,209920,74092,73166,-0.07,0.75,-1.25
Brooklyn,2631580,2630588,777516,779715,118433,118804,-0.04,0.28,0.31
Manhattan,1629477,1628963,809902,809618,44943,44686,-0.03,-0.04,-0.57
Queens,2323052,2322025,613725,612770,129147,131239,-0.04,-0.16,1.62
Staten Island,494956,494796,124064,123564,25670,25517,-0.03,-0.40,-0.60
NYC,8483844,8480225,2533561,2535587,392285,393412,-0.04,0.08,0.29


B15012 'fields of bachelor's degrees reported' NYC: 2,772,579 | PUMS first majors 2,535,587 | first + second majors 2,847,345


In [13]:
# Margins of error with the 80 replicate weights (SDR): SE = sqrt(4/80 * sum((rep - est)^2)); 90% MOE = 1.645 * SE
REPS = [f"PWGTP{i}" for i in range(1, 81)]
def moe(x):
    est = x.w.sum(); rep = x[REPS].astype(int).sum().to_numpy()
    return est, 1.645 * np.sqrt(4 / 80 * ((rep - est) ** 2).sum())
rows = {"NYC bachelor's, age 22-27": b, "  ... Computer Science (2102)": b[b.FOD1P == "2102"],
        "  ... Nursing (6107)": b[b.FOD1P == "6107"], "  ... a median-size field": b[b.FOD1P == n.sort_values().index[len(n) // 2]],
        "NYC associate's, age 22-27": g[g.degree == "associate"]}
m = pd.DataFrame({k: dict(zip(["estimate", "moe90"], moe(v)), survey_rows=len(v)) for k, v in rows.items()}).T
m["moe90_%"] = (m.moe90 / m.estimate * 100).round(0)
display(m.round(0))

,estimate,moe90,survey_rows,moe90_%
"NYC bachelor's, age 22-27",303668.0,5502.0,12346.0,2.0
... Computer Science (2102),16990.0,1686.0,728.0,10.0
... Nursing (6107),6754.0,894.0,306.0,13.0
... a median-size field,808.0,296.0,29.0,37.0
"NYC associate's, age 22-27",47292.0,2359.0,1727.0,5.0


**Tie-out findings**
- **Population:** PUMS is within −0.03% to −0.07% of published B01003 in every borough (NYC 8,480,225 vs 8,483,844) .
- **Bachelor's or higher, 25+:** NYC +0.08% (borough range −0.40% to +0.75%) vs B15003 . Associate's, 25+: NYC +0.29% (−1.25% to +1.62%) .
- **B15012 is a different count:** it reports fields of bachelor's degrees (2,772,579), which is between first majors only (2,535,587) and first + second majors (2,847,345). Not a clean tie; the table's own rules for second majors aren't in the file. Use B15003 as the tie-out.
- **Margins of error** (90%, replicate weights): all NYC bachelor's 22–27 ±2%; Computer Science ±10%; Nursing ±13%; a median-size field (29 rows) ±37%; associate's 22–27 ±5%. A 30-row floor still means ±35–40% for the smallest fields shown. Every ACS number on the dashboard needs its MOE, and small fields should be grouped up.

## 8. Fitness for purpose: a first look at the three uses

### 8a. Underemployment of young bachelor's holders (crowded-field rule)

In [14]:
BA_OK = {"Bachelor's degree", "Master's degree", "Doctoral or professional degree"}
e = emp[emp.age.between(22, 27)].copy()
e["degree"] = r.loc[e.index, "degree"]
for d_ in ["bachelor", "associate"]:
    x = e[e.degree == d_]; known = x[x.entry.notna()]
    need = "a bachelor's or more" if d_ == "bachelor" else "less than an associate's"
    if d_ == "bachelor":
        share = (known.w * ~known.entry.isin(BA_OK)).sum() / known.w.sum()
    else:
        share = (known.w * known.entry.isin({"High school diploma or equivalent", "No formal educational credential"})).sum() / known.w.sum()
    print(f"{d_}, 22-27, employed: entry level known for {(x.w * x.entry.notna()).sum() / x.w.sum():.0%}; "
          f"of those, in jobs whose BLS typical entry is below the degree: {share:.1%}")

# Pay: full-time, year-round workers (35+ hours, 50+ weeks), wages in 2024 dollars
r["wage_2024"] = pd.to_numeric(r.WAGP, errors="coerce") * r.ADJINC.astype(int) / 1e6
ft = r[(pd.to_numeric(r.WKHP, errors="coerce") >= 35) & (pd.to_numeric(r.WKWN, errors="coerce") >= 50) & (r.wage_2024 > 0)]
def wmedian(x):
    x = x.sort_values("wage_2024"); return x.wage_2024[x.w.cumsum() >= x.w.sum() / 2].iloc[0]
pay = pd.DataFrame({f"age {a}-{b_}": ft[ft.age.between(a, b_)].groupby("degree").apply(wmedian, include_groups=False) for a, b_ in [(22, 27), (35, 44)]}).round(-2)
print("\nMedian wage, full-time year-round NYC residents, 2024 dollars:"); display(pay)

bachelor, 22-27, employed: entry level known for 82%; of those, in jobs whose BLS typical entry is below the degree: 38.8%
associate, 22-27, employed: entry level known for 85%; of those, in jobs whose BLS typical entry is below the degree: 69.2%



Median wage, full-time year-round NYC residents, 2024 dollars:


,age 22-27,age 35-44
degree,,
associate,43600.0,61100.0
bachelor,73500.0,102500.0
below associate,37800.0,48900.0
graduate,78200.0,121800.0


### 8b. Major → job check (decision D2): what do young NYC graduates with a given major do?

In [15]:
OCC_TITLE = occ.set_index("occp").title.str.strip().to_dict()
def top_jobs(fod, k=6):
    x = e[(e.degree == "bachelor") & (e.FOD1P == fod)]
    t = x.groupby("OCCP").w.sum().sort_values(ascending=False)
    out = (t / t.sum() * 100).round(1).head(k)
    return pd.DataFrame({"job": out.index.map(OCC_TITLE), "share_%": out.values, "entry": out.index.map(lambda o: x[x.OCCP == o].entry.iloc[0])}), len(x)
for f in ["2102", "6107", "5200", "6203", "3401"]:
    t, nrows = top_jobs(f)
    print(f"\n{FOD[f]} ({f}): employed bachelor's 22-27, {nrows} survey rows")
    display(t)


Computer Science (2102): employed bachelor's 22-27, 595 survey rows


,job,share_%,entry
0,Software developers,45.0,Bachelor's degree
1,Computer programmers,4.1,Bachelor's degree
2,Computer support specialists,3.3,NaN
3,Management analysts,2.4,Bachelor's degree
4,Teaching assistants,1.7,NaN
5,Data entry keyers,1.5,High school diploma or equivalent



Nursing (6107): employed bachelor's 22-27, 250 survey rows


,job,share_%,entry
0,Registered nurses,66.9,Bachelor's degree
1,Medical assistants,3.1,Postsecondary nondegree award
2,Nursing assistants,2.3,Postsecondary nondegree award
3,Elementary and middle school teachers,1.8,Bachelor's degree
4,Other mathematical science occupations,1.7,NaN
5,"Hotel, motel, and resort desk clerks",1.5,High school diploma or equivalent



Psychology (5200): employed bachelor's 22-27, 528 survey rows


,job,share_%,entry
0,Elementary and middle school teachers,5.6,Bachelor's degree
1,"Secretaries and administrative assistants, except legal, medical, ...",4.8,High school diploma or equivalent
2,"Managers, all other",4.2,NaN
3,Receptionists and information clerks,3.6,High school diploma or equivalent
4,Customer service representatives,2.7,High school diploma or equivalent
5,Registered nurses,2.4,Bachelor's degree



Business Management And Administration (6203): employed bachelor's 22-27, 344 survey rows


,job,share_%,entry
0,"Securities, commodities, and financial services sales agents",6.0,Bachelor's degree
1,"Managers, all other",5.9,NaN
2,Financial and investment analysts,3.4,Bachelor's degree
3,Market research analysts and marketing specialists,3.1,Bachelor's degree
4,Receptionists and information clerks,2.7,High school diploma or equivalent
5,Management analysts,2.7,Bachelor's degree



Liberal Arts (3401): employed bachelor's 22-27, 87 survey rows


,job,share_%,entry
0,Financial and investment analysts,6.8,Bachelor's degree
1,Preschool and kindergarten teachers,4.8,NaN
2,"Secretaries and administrative assistants, except legal, medical, ...",4.7,High school diploma or equivalent
3,"Social workers, all other",4.6,Bachelor's degree
4,Customer service representatives,4.4,High school diploma or equivalent
5,Market research analysts and marketing specialists,3.8,Bachelor's degree


### 8c. Junior vs senior (brief O1): can ACS give the share of each job's workers who are graduation-age?

In [16]:
jw = emp.groupby("SOCP").apply(lambda x: pd.Series({"rows": len(x), "rows_22_29": int(x.age.between(22, 29).sum()),
                                                    "share_22_29_%": (x.w * x.age.between(22, 29)).sum() / x.w.sum() * 100}), include_groups=False)
print(f"{len(jw)} SOCP codes; with >= 30 rows aged 22-29: {(jw.rows_22_29 >= 30).sum()}, holding "
      f"{emp[emp.SOCP.isin(jw[jw.rows_22_29 >= 30].index)].w.sum() / emp.w.sum():.0%} of NYC workers")
display(jw[jw.rows >= 300].sort_values("share_22_29_%").iloc[[0, 1, 2, -3, -2, -1]].round(1)
        .assign(job=lambda t: t.index.map(lambda s: OCC_TITLE.get(emp.loc[emp.SOCP == s, 'OCCP'].iloc[0]))))

524 SOCP codes; with >= 30 rows aged 22-29: 182, holding 87% of NYC workers


,rows,rows_22_29,share_22_29_%,job
SOCP,,,,
533052,535.0,24.0,5.3,"Bus drivers, transit and intercity"
19303X,408.0,20.0,5.5,Other psychologists
372012,1622.0,104.0,6.5,Maids and housekeeping cleaners
1520XX,675.0,263.0,40.7,Other mathematical science occupations
131161,862.0,324.0,40.9,Market research analysts and marketing specialists
273031,363.0,136.0,45.0,Public relations specialists


**Fitness findings**
- **Underemployment works.** Of employed NYC bachelor's holders aged 22–27 (82% with a clear entry level), 38.8% work in jobs whose BLS typical entry is below a bachelor's. The New York Fed's national figure for recent graduates is about 41%, so this is in range. For associate's holders, 69.2% are in jobs needing only high school or less, but 41% of them are still students, so this mostly measures student jobs. Underemployment for associate's should be measured on those not in school.
- **Pay works.** Median wage, full-time year-round, 2024 dollars, ages 22–27: below associate's $37.8k, associate's $43.6k, bachelor's $73.5k, graduate $78.2k. At 35–44: $48.9k / $61.1k / $102.5k / $121.8k.
- Major → job check works for bachelor's, at the field level. Computer Science → software developers 45%; Nursing → registered nurses 67%; Psychology spreads out (no job above 6%, many clerical); Business Management spreads over finance, management and marketing. Liberal Arts BA holders (87 rows) also spread out with no dominant job. This is the evidence D2 needs: some majors lead to one job, others to a wide field.
- **Junior vs senior (O1): feasible.** 182 job codes have ≥ 30 workers aged 22–29, covering 87% of NYC workers. The share aged 22–29 runs from ~5% (bus drivers, maids) to 40–45% (market research analysts, PR specialists). That's a usable "how young is this job's workforce" measure, a candidate for one of the three O1 estimates.

## 9. Issues list and proposed staging table

### 8. Fitness for purpose

| Question | Fit | Why |
|---|---|---|
| Q2: what NYC bachelor's grads actually do, by major | Use | Field of degree × job × age on one row, 12k graduation-age rows. Field level only (FOD), with MOEs |
| **Q2: same for associate's grads** | no | ACS has no field of degree below a bachelor's. Only "associate's holders in general" |
| Q4: what young grads earn, by degree level (and by field for big fields) | yes | Wages in 2024 dollars, full-time year-round filter |
| **S11 crowded field: underemployment** | yes | 82% of young bachelor's workers get a BLS entry level; group rules cover most of the rest |
| O1 junior vs senior: share of workers aged 22–29 by job | as one estimate | 182 jobs with ≥ 30 young rows, 87% of workers |
| D2: check major → job links | bachelor's only | Needs the hand-built FOD → CIP table (O5) |
| Trends year by year | partial | A 5-year file is one pooled period. Year-by-year needs the 1-year files (smaller samples) |
| Everyone working in NYC | partial | Resident-based; NJ/CT commuters are in other state files |

**Recommendation for v1:** use PUMS as the outcomes source: what NYC residents with each degree level (and each bachelor's field) do and earn, underemployment, and the young-worker share by job. Always show the MOE; group fields below 30 rows; label "NYC residents, 2020–2024 average". Don't use it as allocation weights (D2 recommendation stands).

### 9. Issues list

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | No field of degree below a bachelor's | all associate's holders | Associate's outcomes by degree level only; say so on the page |
| 2 | No public FOD → CIP crosswalk | all bachelor's+ | Hand-build `ref_fod_cip` (brief O5), many-to-many to CIP 4-digit series; needs a decision |
| 3 | `SOCP` groups (`X`/`Y`, broad groups) | 37.6% of workers | `ref_socp_soc`: exact → group with one level → BLS parent aggregate → mixed group (OEWS NY biggest job) |
| 4 | Heavy imputation (FOD 25%, OCC 24%, wages 34%) | ~¼ of rows | Keep the flags as booleans; sensitivity check without imputed rows for D2 |
| 5 | Small cells | fields with < 30 rows | MOE from replicate weights on every estimate; fields below 30 rows grouped to the FOD group ("Science and Engineering" …) |
| 6 | 5-year pooled; weights = period average | all | One "2020–2024" period; no single-year totals from this file |
| 7 | Pay in each year's dollars | `WAGP`, `PERNP` | `× ADJINC / 1e6` → 2024 dollars in staging |
| 8 | Resident-based; NJ/CT commuters absent | NYC workforce | Label "NYC residents"; NJ/CT files only if a "who works in NYC" view is needed |
| 9 | `NAICSP` Census groups (`3M`, `4M`, `722Z` …) | < 1% of workers | Map to NAICS sector by first 2 characters; `3M → 31-33`, `4M` → "trade, not specified" |
| 10 | Still-enrolled graduates | 41% of young associate's, 23% of bachelor's | `enrolled` flag; outcome measures computed on those not in school |
| 11 | 2020 data collection was disrupted | 2020 rows | Census already adjusted the weights; footnote only |

### Proposed staging table

**`stg_census__acs_pums_person`**
- **Grain / key:** one person: `(serialno, sporder)`; NYC residents aged 16+, 2020–2024 pooled
- **Columns:** `serialno`, `sporder`, `response_year`, `puma`, `borough`, `is_group_quarters`, `age`, `sex`, `education_code` (SCHL), `degree_level` (`below_associate`/`associate`/`bachelor`/`graduate`), `enrolled`, `fod1`, `fod2`, `employment_status`, `class_of_worker`, `occp`, `socp`, `naicsp`, `wage_2024usd`, `earnings_2024usd`, `usual_hours`, `weeks_worked`, `works_in_nyc`, `fod_imputed`, `occ_imputed`, `wage_imputed`, `weight`, `rep_weight_1 … rep_weight_80`
- **Filters:** NYC PUMAs, age ≥ 16 (row count below). Reference tables alongside: `ref_fod` (code list), `ref_socp_soc` (issue 3), `ref_fod_cip` (issue 2, hand-built)

In [17]:
# Row accounting for the proposed staging filter, recomputed rather than typed
stg = r[r.age >= 16]
assert not stg.duplicated(["SERIALNO", "SPORDER"]).any()
print(f"NY State rows: {n_state:,} -> NYC residents: {len(r):,} -> staging (age 16+): {len(stg):,} "
      f"| of which graduation-age (22-27) associate's or bachelor's: {len(g):,}")

NY State rows: 984,474 -> NYC residents: 326,729 -> staging (age 16+): 279,311 | of which graduation-age (22-27) associate's or bachelor's: 14,073
